# Deezer vs Rekordbox — Comparaison brute

In [149]:
import sys
sys.path.insert(0, "..")

from server.deezer.extractor import DeezerExtractor
from worker.rekordbox.extractor import RekordboxExtractor

deezer = DeezerExtractor("656772321")
rb = RekordboxExtractor()

In [150]:
# --- Extract Deezer ---
# {playlist_title: [track, ...]}
deezer_playlists = {}
for pl in deezer.get_playlists():
    deezer_playlists[pl["title"]] = deezer.get_tracks(pl["id"])

print(f"{len(deezer_playlists)} playlists Deezer chargées")
for name, tracks in deezer_playlists.items():
    print(f"  {name}  ({len(tracks)} tracks)")

15 playlists Deezer chargées
  W - Classic/Min. Techno  (31 tracks)
  W - Deep House  (43 tracks)
  W - Downtempo  (13 tracks)
  W - Electro Brut  (9 tracks)
  W - French Touch  (46 tracks)
  W - Hard/Dark Techno  (38 tracks)
  W - Hard Groove  (0 tracks)
  W - Melodic Techno  (27 tracks)
  W - Misc. Tracks  (12 tracks)
  W - Nu Disco  (39 tracks)
  W - Psytrance  (40 tracks)
  W - Tech House  (53 tracks)
  W - Trance Techno  (156 tracks)
  W - UK Garage  (24 tracks)
  W - UK House  (64 tracks)


In [151]:
# --- Mapping Deezer playlist → tag RB ---
from difflib import get_close_matches

def normalize_tag(s: str) -> str:
    return s.lower().replace("-", " ").replace("_", " ").strip()

def deezer_to_rb_tag(playlist_title: str, rb_tags: list[str]) -> str:
    candidate = playlist_title.removeprefix("W - ")
    # 1. Match exact après normalisation
    norm_candidate = normalize_tag(candidate)
    for tag in rb_tags:
        if normalize_tag(tag) == norm_candidate:
            return tag
    # 2. Fallback Levenshtein (seuil ~0.85 ≈ distance 3 sur noms courts)
    matches = get_close_matches(norm_candidate, [normalize_tag(t) for t in rb_tags], n=1, cutoff=0.85)
    if matches:
        # Retrouve le tag original correspondant
        for tag in rb_tags:
            if normalize_tag(tag) == matches[0]:
                return tag
    return candidate  # aucun match, retourne tel quel

# Affiche les correspondances
print("Correspondances Deezer → Rekordbox :")
for pl_name in deezer_playlists:
    rb_tag = deezer_to_rb_tag(pl_name, list(rb_by_tag.keys()))
    found = rb_tag in rb_by_tag
    status = "OK" if found else "MANQUANT dans RB"
    print(f"  {pl_name}  →  '{rb_tag}'  [{status}]")

Correspondances Deezer → Rekordbox :
  W - Classic/Min. Techno  →  'Classic/Min. Techno'  [OK]
  W - Deep House  →  'Deep House'  [OK]
  W - Downtempo  →  'Downtempo'  [OK]
  W - Electro Brut  →  'Electro Brut'  [OK]
  W - French Touch  →  'French Touch'  [OK]
  W - Hard/Dark Techno  →  'Hard/Dark Techno'  [OK]
  W - Hard Groove  →  'Hard Groove'  [MANQUANT dans RB]
  W - Melodic Techno  →  'Melodic Techno'  [OK]
  W - Misc. Tracks  →  'Misc. Tracks'  [OK]
  W - Nu Disco  →  'Nu-Disco'  [OK]
  W - Psytrance  →  'Psytrance'  [OK]
  W - Tech House  →  'Tech House'  [OK]
  W - Trance Techno  →  'Trance Techno'  [OK]
  W - UK Garage  →  'UK Garage'  [OK]
  W - UK House  →  'UK House'  [OK]


In [152]:
from server.deezer.sync_checker import check_sync

report = check_sync(deezer_playlists, rb_by_tag)
print(report.summary())


[DOWNLOAD_NEEDED] (1)
  - Collect 200 — Pull Up  (playlist: W - UK House)


In [153]:
# Cherche dans quelle playlist Deezer se trouve un titre
search = "where have you been"
for pl_name, tracks in deezer_playlists.items():
    for t in tracks:
        if search in t["title"].lower():
            print(f"{pl_name}  →  {t['artist']['name']} — {t['title']}")